In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from imblearn.over_sampling import SMOTENC

In [2]:
bank= pd.read_csv('/kaggle/input/datasets/prahazra/bank-data/European_Bank.csv')
bank.sample(5)

,Year,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
4183,2025,15625592,Sal,486,France,Male,26,2,0.00,2,1,1,31399.40,0
8894,2025,15631222,Cattaneo,485,France,Female,39,2,75339.64,1,1,1,70665.16,0
3115,2025,15582066,Maclean,561,France,Male,21,4,0.00,1,1,1,36942.35,0
1257,2025,15647402,Wan,628,France,Female,38,3,0.00,2,1,1,48924.73,0
8395,2025,15586069,Abernathy,560,France,Female,30,0,108883.29,1,1,0,27914.95,0


In [3]:
bank.shape

(10000, 14)

# General Preproccessing

Cheking if there are any duplicate values ornot. If not then procede to elemination of the non-important features.

Remove non-informative features

In [4]:
bank.duplicated().sum()

np.int64(0)

In [5]:
bank.drop(columns=['Year','Surname','CustomerId'], inplace=True)
bank.sample(5)

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
1877,770,Spain,Male,55,9,63127.41,2,1,0,185211.28,1
2567,553,France,Male,42,1,0.00,2,0,0,23822.04,0
3225,850,Germany,Female,32,0,116968.91,1,0,0,175094.62,0
7985,696,Germany,Female,27,2,96129.32,2,1,1,5983.70,0
4579,534,France,Male,52,1,0.00,3,1,1,104035.41,1


Check for missing values in the data

In [6]:
bank.isnull().sum()

CreditScore        0
Geography          0
Gender             0
Age                0
Tenure             0
Balance            0
NumOfProducts      0
HasCrCard          0
IsActiveMember     0
EstimatedSalary    0
Exited             0
dtype: int64

In [7]:
for col in bank.columns:
  if pd.api.types.is_numeric_dtype(bank[col]):
    z_scores = np.abs(stats.zscore(bank[col]))
    threshold = 3
    outliers = np.where(z_scores > threshold)
    print(f'Number of outliers in {col}: {len(outliers[0])}')
  else:
    print(f"Skipping non-numeric column '{col}' for outlier detection.")

Number of outliers in CreditScore: 8
Skipping non-numeric column 'Geography' for outlier detection.
Skipping non-numeric column 'Gender' for outlier detection.
Number of outliers in Age: 133
Number of outliers in Tenure: 0
Number of outliers in Balance: 0
Number of outliers in NumOfProducts: 60
Number of outliers in HasCrCard: 0
Number of outliers in IsActiveMember: 0
Number of outliers in EstimatedSalary: 0
Number of outliers in Exited: 0


We have outliers in 'Age' and 'NumberOfProducts' column. Let's look at them in more details.

In [8]:
bank['Age'].describe()

count    10000.000000
mean        38.921800
std         10.487806
min         18.000000
25%         32.000000
50%         37.000000
75%         44.000000
max         92.000000
Name: Age, dtype: float64

We can see that there is no impossible entry in the 'Age' column, and for the given problem we have to target all the customer regarding their age especially if the older customer make the most deposits in term of pensions etc. So, removing those outliers will affect the model prediction. On the other hand if the older customer don't cotribute much to the bank revenue then removing them will be good approach.

In [9]:
bank['NumOfProducts'].describe()

count    10000.000000
mean         1.530200
std          0.581654
min          1.000000
25%          1.000000
50%          1.000000
75%          2.000000
max          4.000000
Name: NumOfProducts, dtype: float64

Since the third quantile is 2 and max is 4 for the 'NumOfProducst' column, the detected outliers can't be considered as outliers that will affect the model. So, we will nor remove them.

In [10]:
bank.dtypes

CreditScore          int64
Geography           object
Gender              object
Age                  int64
Tenure               int64
Balance            float64
NumOfProducts        int64
HasCrCard            int64
IsActiveMember       int64
EstimatedSalary    float64
Exited               int64
dtype: object

In [11]:
bank['Exited'].value_counts()

Exited
0    7963
1    2037
Name: count, dtype: int64

# Feature Engineering

In [12]:
def create_basic_features(df):
  # Age Group
  df['AgeGroup'] = pd.cut(df['Age'], bins = [0, 19, 35, 60, 120], labels = ['Teenager', 'Young', 'Mid-age', 'Old'])

  # Binary Balance
  df['IsZeroBalance'] = df['Balance'].apply(lambda x: 1 if x > 0 else 0)

  # Credit score
  df['CreditScoreRating'] = pd.cut(df['CreditScore'], bins=[300, 579, 669, 739, 799, 850], labels=['Poor', 'Fair', 'Good', 'Very Good', 'Excellent'])

  #Binary Product Indicator
  df['HasMultipleProduct'] = df['NumOfProducts'].apply(lambda x: 1 if x > 1 else 0)
  return df

def create_intermediate_features(df):
  # Balance-to-Salary Ratio
  df['BalanceSalaryRatio']= df['Balance']/df['EstimatedSalary']

  # Tenure-Age Ratio
  df['TenureAgeRatio']= df['Tenure']/df['Age']

  # Credit Score per Age
  df['CreditScoreAgeRatio']= df['CreditScore']/df['Age']

  # Estimated Monthly Salary
  df['EstimatedMonthlySalary']= df['EstimatedSalary']/12
  return df

def create_advanced_features(df):
  # Age Balance Interaction
  df['AgeBalanceInteraction']= df['Age'] * df['Balance']

  # Product Density
  # Handle division by zero for Tenure: if Tenure is 0, ProductDensity is set to 0
  df['ProductDensity']= df.apply(lambda row: row['NumOfProducts'] / row['Tenure'] if row['Tenure'] != 0 else 0, axis=1)

  # Loyalty-Engagement Score
  df['LoyalityScore']= df['Tenure'] * df['IsActiveMember']

  # Wealth Signifier
  median_balance = df['Balance'].median()
  median_salary = df['EstimatedSalary'].median()
  df['WealthSignifier'] = ((df['Balance'] > median_balance) & (df['EstimatedSalary'] > median_salary)).astype(int)
  return df

def create_additional_features(df):
    df['TenureRiskGroup'] = pd.cut(df['Tenure'], bins=[-1, 2, 9, np.inf], labels=['Early_HighRisk', 'Mid_Stable', 'Late_HighRisk'])
    df['MidTierDanger'] = df['Balance'].between(100000, 150000).astype(int)
    df['GermanPassiveWealth'] = ((df['Geography'] == 'Germany') & (df['IsActiveMember'] == 0) & (df['Balance'] > 0)).astype(int)
    df['HighRiskGermanCohort'] = ((df['Geography'] == 'Germany') & 
                              (df['Age'] >= 45) & 
                              (df['IsActiveMember'] == 0) & 
                              (df['NumOfProducts'].isin([1, 3, 4]))).astype(int)
    df['BalancePerProduct'] = df['Balance'] / df['NumOfProducts']
    df['CardButInactive'] = ((df['HasCrCard'] == 1) & (df['IsActiveMember'] == 0)).astype(int)
    df['CardAndActive'] = ((df['HasCrCard'] == 1) & (df['IsActiveMember'] == 1)).astype(int)
    df['TeenZeroBalance'] = ((df['Age'] <= 19) & (df['Balance'] == 0)).astype(int)
    df['ActiveZeroBalance'] = ((df['Balance'] == 0) & (df['IsActiveMember'] == 1)).astype(int)
    df['Female_Germany'] = ((df['Gender'] == 'Female') & (df['Geography'] == 'Germany')).astype(int)
    return df

In [13]:
bank= create_basic_features(bank)
bank= create_intermediate_features(bank)
bank= create_advanced_features(bank)
bank= create_additional_features(bank)

In [14]:
bank.columns

Index(['CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance',
       'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary',
       'Exited', 'AgeGroup', 'IsZeroBalance', 'CreditScoreRating',
       'HasMultipleProduct', 'BalanceSalaryRatio', 'TenureAgeRatio',
       'CreditScoreAgeRatio', 'EstimatedMonthlySalary',
       'AgeBalanceInteraction', 'ProductDensity', 'LoyalityScore',
       'WealthSignifier', 'TenureRiskGroup', 'MidTierDanger',
       'GermanPassiveWealth', 'HighRiskGermanCohort', 'BalancePerProduct',
       'CardButInactive', 'CardAndActive', 'TeenZeroBalance',
       'ActiveZeroBalance', 'Female_Germany'],
      dtype='object')

In [15]:
bank.to_csv("Final_bank_data.csv", index= False)